## **Plotting directionality**

The rate axis of `cr` has length `2 * len(rates)`, not `len(rates)`. That doubling is **not** extra rate resolution: it encodes the *direction* in which a spectrotemporal pattern sweeps across frequency. `aud2cor` runs every rate filter twice, once per sign, and stores the two results in separate halves:

```python
for sgn in [1, -1]:
    ...
    col = rdx + (num_rates if sgn == 1 else 0)
```

- columns `0 .. len(rates)-1` come from the `sgn = -1` branch
- columns `len(rates) .. 2*len(rates)-1` come from the `sgn = +1` branch

By default `plot_cr_projection` **averages the two halves together**, so it is direction-blind: an upward and a downward sweep of the same rate and scale produce the same picture. Passing `signed_rates=True` instead lays the halves side by side on a single **signed** rate axis running `-rates[::-1]` through `+rates`, with a dashed line marking the boundary:

```python
plot_cr_projection(cr, rates, scales=scales, frequencies=freqs, signed_rates=True)
```

The Scale-Rate and Rate-Frequency panels then span all `2 * len(rates)` channels, and each panel keeps a single colour scale across both halves. That last part matters: plotting one direction per figure lets each auto-scale independently, which makes the weaker direction look as strong as the dominant one.


In [1]:
import numpy as np
import matplotlib.pyplot as plt

from neural_audio.wav2aud import wav2aud
from neural_audio.aud2cor import aud2cor
from neural_audio.utils.mathfuncs import gen_ripple
from neural_audio.utils.visualize_outputs import cr_projections, plot_cr_projection

In [ ]:


# An 8 x 8 grid and a 0.5 s ripple keep each cr small (~33 MB) while still resolving the peak.
rates  = 2 ** np.linspace(np.log2(0.5), np.log2(32), 8)   # temporal rates [Hz]
scales = 2 ** np.linspace(np.log2(1/5), np.log2(8), 8)    # spectral scales [cyc/oct]

# The signed axis plot_cr_projection labels against, for reading peaks off in code.
signed = np.concatenate([-rates[::-1], rates])

# Two stimuli with the same rate and scale magnitude, opposite sweep direction.
directions = {"downward (rate > 0)": +rates[4], "upward (rate < 0)": -rates[4]}

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for row, (name, rate) in zip(axes, directions.items()):
    _, freqs, aud = wav2aud(gen_ripple(rate=rate, scale=1.65, duration=0.5))
    cr = aud2cor(aud, rates=rates, scales=scales)

    plot_cr_projection(cr, rates, scales=scales, frequencies=freqs,
                       axes=row, signed_rates=True)
    row[0].set_ylabel(f"{name}\nScale [cyc/oct]")

    # scale_rate is already on the signed axis, so index `signed` with its argmax directly
    scale_rate, _, _ = cr_projections(cr, rates, signed_rates=True)
    peak = signed[scale_rate.mean(axis=0).argmax()]
    print(f"{name:22s} stimulus {rate:+.2f} Hz  ->  Scale-Rate peak at {peak:+.2f} Hz")
    del cr  # ~33 MB each, so don't hold both at once

fig.suptitle("signed_rates=True: the peak sits on the side matching the sweep direction",
             fontweight="bold")
fig.tight_layout()
plt.show()


##### **Reading the result**

```
downward (rate > 0)    stimulus +5.38 Hz  ->  Scale-Rate peak at +5.38 Hz
upward (rate < 0)      stimulus -5.38 Hz  ->  Scale-Rate peak at -5.38 Hz
```

Each stimulus recovers its own rate *and* its own sign. The Scale-Rate peak sits at `+5.4 Hz, 1.6 cyc/oct` for the downward ripple and mirrors to `-5.4 Hz` for the upward one, and the Rate-Frequency panels show the same band flipping from one side of the divider to the other. Averaging the halves, as the default does, would place both stimuli in the same spot and dilute the peak.

A few things worth knowing:

- **The sign convention is a property of this setup, not a universal fact.** Which physical direction lands on the positive half depends on the sign convention of the rate vector *and* on how the stimulus was generated. Measure it as above rather than assuming it carried over.
- **`signed` is already ordered**, so `signed[scale_rate.mean(0).argmax()]` reads the peak directly. Do not also apply the column-reordering used internally: reordering twice scrambles the negative half while leaving the positive half untouched, which produces a plausible-looking but wrong answer.
- **Margins are handled.** The rate axis is never padded by `temporal_margin` or `spectral_margin`, so signed rate ticks take no offset while the frequency ticks keep their existing `+ dM` shift. `signed_rates=True` works unchanged with either margin set.
- **The default is still the right choice for summary figures**, where comparability across stimuli matters more than direction. Reach for `signed_rates=True` when direction is the thing being characterised, since natural sounds are not symmetric in sweep direction.


In [8]:
def gen_corf(tune_scale: int, filt_len: int, channels_per_oct: int, PASS=None):
    """ 
    Calculates frequency-domain transfer function for a spectral filter based on
    the given parameters.

    A scale (`octave_offsets`) is calculated: the relative position along the filter length
    is mapped to a frequency axis of one octave, then divided by two to span half an octave 
    instead. The higher the number of channels per octave, the higher your number of 'steps'
    (units) and hence the higher the frequency resolution. This scale is divided by the
    number of cycles per octave, producing units of 'channels per 0.5 cycles'.

    Two functions can be used to define the filter response, either a Gabor function or the
    negative second derivative of a Gaussian, selected via `func_type`. Importantly, it is
    their frequency-domain forms that are used.

    For a lowpass filter, everything up to the maximum is flattened to 1 (ideal pass). Likewise, 
    for highpass filtering, everything from the maximum until the end is flattened to 1. In both
    cases, the sum of the transfer function is readjusted to preserve the filter's gain acorticogramoss all
    passbands. If bandpass behavior is specified, the transfer function remains unchanged. 

    :param tune_scale: (Spectral) modulation rate in cycles per octave
    :type tune_scale: int

    :param filt_len: Filter length, ideally a power of 2
    :type filt_len: int

    :param channels_per_oct: Frequency resolution in channels per octave
    :type channels_per_oct: int

    :param PASS: Array of shape `[idx, upper_bound]` where `idx` denotes index and `upper_bound` is the maximum
        value of this index. Dictates passband.
        If none supplied, defaults to bandpass (`[2, 3]`). Possible values and outcomes:
            - `idx = 1`: lowpass
            - `1 < idx < upper_bound`: bandpass
            - `idx = upper_bound`: highpass
    :type PASS: np.ndarray, optional, default=None

    :returns: np.ndarray - Filter transfer function (in frequency domain)

    .. note:: The original MATLAB implementation of this function included an option to use a Gabor (rather than a Gaussian)
        function to define filter behavior. Since it is unclear how modifying this parameter would affect the biological or 
        mathematical validity of the model, this option has been removed.
    """

    octave_offsets = np.arange(filt_len)/filt_len * channels_per_oct/2/abs(tune_scale)
    print("octave_offsets:", octave_offsets, "\n ---- \n")

    # Gaussian function used
    octave_offsets = octave_offsets**2
    freq_tf = octave_offsets*np.exp(1-octave_offsets)

    if PASS is not None:
        if PASS[0] == 1:
            max_idx = np.argmax(freq_tf)
            s = np.sum(freq_tf)
            freq_tf[:max_idx] = 1
            freq_tf = freq_tf/np.sum(freq_tf)*s

        elif PASS[0] == PASS[1]:
            max_idx = np.argmax(freq_tf)
            s = np.sum(freq_tf)
            freq_tf[max_idx+1:] = 1
            freq_tf = freq_tf/np.sum(freq_tf)*s

    return freq_tf

In [9]:
half_rates  = 2 ** np.linspace(np.log2(0.5), np.log2(128), 16)
half_scales = 2 ** np.linspace(np.log2(1/5), np.log2(10), 16)
bp = 1
num_rates = len(half_rates)

N, M = (750, 128)  # (time frames, frequency channels)
# FFT padding
N_pad = int(2 ** np.ceil(np.log2(N)))
M_pad = int(2 ** np.ceil(np.log2(M)))

fps = 1000 / 4  # frames per second, with the default frame_length of 4 ms
channels_per_oct = 20 if M == 95 else 24
num_rates, num_scales = len(half_rates), len(half_scales)



In [10]:
scales_to_plot = [half_scales[0], half_scales[10], half_scales[-1]]  # or whichever scales you want

for scale in scales_to_plot:
    sdx = np.argmin(np.abs(half_scales - scale))
    bp = 1
    H = gen_corf(half_scales[sdx], M_pad, channels_per_oct, PASS=[sdx + 1 + bp, num_scales + bp * 2])

octave_offsets: [ 0.       0.46875  0.9375   1.40625  1.875    2.34375  2.8125   3.28125
  3.75     4.21875  4.6875   5.15625  5.625    6.09375  6.5625   7.03125
  7.5      7.96875  8.4375   8.90625  9.375    9.84375 10.3125  10.78125
 11.25    11.71875 12.1875  12.65625 13.125   13.59375 14.0625  14.53125
 15.      15.46875 15.9375  16.40625 16.875   17.34375 17.8125  18.28125
 18.75    19.21875 19.6875  20.15625 20.625   21.09375 21.5625  22.03125
 22.5     22.96875 23.4375  23.90625 24.375   24.84375 25.3125  25.78125
 26.25    26.71875 27.1875  27.65625 28.125   28.59375 29.0625  29.53125
 30.      30.46875 30.9375  31.40625 31.875   32.34375 32.8125  33.28125
 33.75    34.21875 34.6875  35.15625 35.625   36.09375 36.5625  37.03125
 37.5     37.96875 38.4375  38.90625 39.375   39.84375 40.3125  40.78125
 41.25    41.71875 42.1875  42.65625 43.125   43.59375 44.0625  44.53125
 45.      45.46875 45.9375  46.40625 46.875   47.34375 47.8125  48.28125
 48.75    49.21875 49.6875  50.1562